# Klaatch Data Stratification for Cross-Validation

## Overview

This notebook performs stratified k-fold assignment for cross-validation on the Klaatch dataset. The stratification ensures that:
- All records from the same participant (klaatch_id) are assigned to the same fold
- Folds are balanced in terms of total number of samples
- No data leakage occurs between training and test sets

## Methodology

1. **Load Data**: Extract data from MySQL database
2. **Extract Participant IDs**: Parse klaatch_id from message_id
3. **Balanced Assignment**: Use greedy algorithm to assign participants to folds
4. **Validation**: Verify no participant appears in multiple folds
5. **Update Database**: Store fold assignments back to MySQL

## Configuration

- **Table**: `stratified_female`
- **Number of Folds**: 5
- **Grouping**: By klaatch_id (participant)
- **Strategy**: Greedy assignment for balanced fold sizes

---

## 1. Import Libraries and Setup

In [ ]:
import pandas as pd
import mysql.connector
import numpy as np
from collections import defaultdict
import os

# Configuration
DB_CONFIG = {
    'host': 'localhost',
    'user': os.getenv('DB_USER', '************'),
    'password': os.getenv('DB_PASSWORD', '**************'),
    'database': 'Audio_features'
}

TABLE_NAME = 'stratified_female'
KLAATCH_ID_COL = 'klaatch_id'
FOLD_COL = 'fold'
N_SPLITS = 5

print("Configuration loaded successfully")
print(f"Target table: {TABLE_NAME}")
print(f"Number of folds: {N_SPLITS}")

## 2. Database Connection and Data Loading

In [ ]:
def create_mysql_connection(config):
    """
    Create and return MySQL database connection.
    
    Args:
        config (dict): Database configuration dictionary
        
    Returns:
        mysql.connector.connection: Database connection object
    """
    try:
        conn = mysql.connector.connect(**config)
        print("✓ Database connection successful")
        return conn
    except mysql.connector.Error as err:
        print(f"✗ Database connection failed: {err}")
        raise


def load_table_to_dataframe(table_name, conn):
    """
    Load MySQL table into pandas DataFrame.
    
    Args:
        table_name (str): Name of the table to load
        conn: MySQL connection object
        
    Returns:
        pd.DataFrame: Table data as DataFrame
    """
    cursor = conn.cursor(dictionary=True)
    query = f"SELECT * FROM {table_name}"
    
    try:
        cursor.execute(query)
        rows = cursor.fetchall()
        df = pd.DataFrame(rows)
        print(f"✓ Loaded {len(df)} records from table '{table_name}'")
        return df
    except mysql.connector.Error as err:
        print(f"✗ Error loading table: {err}")
        raise
    finally:
        cursor.close()

# Create connection and load data
conn = create_mysql_connection(DB_CONFIG)
df = load_table_to_dataframe(TABLE_NAME, conn)

print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

## 3. Stratified Fold Assignment

Using a greedy algorithm to assign participants to folds:
- Sort participants by number of records (descending)
- Assign each participant to the fold with the least total records
- This ensures balanced fold sizes while keeping participant data together

In [ ]:
def assign_folds_balanced(df, group_id_col='message_id', n_splits=5):
    """
    Assign fold numbers ensuring balanced distribution of samples per fold.
    
    Uses a greedy bin-packing algorithm:
    1. Extract klaatch_id from message_id (format: {klaatch_id}_{date})
    2. Count records per klaatch_id
    3. Sort by count (descending) for better balance
    4. Assign each klaatch_id to the fold with fewest total records
    
    Args:
        df (pd.DataFrame): DataFrame with message_id column
        group_id_col (str): Column containing group identifiers
        n_splits (int): Number of folds for cross-validation
        
    Returns:
        pd.DataFrame: DataFrame with added 'klaatch_id' and 'fold' columns
    """
    # Extract klaatch_id from message_id (format: {id}_{date})
    df['klaatch_id'] = df[group_id_col].astype(str).str.split('_').str[0]
    
    # Count records per klaatch_id
    klaatch_counts = df['klaatch_id'].value_counts().reset_index()
    klaatch_counts.columns = ['klaatch_id', 'count']
    
    # Sort by count (descending) for better balance
    klaatch_counts = klaatch_counts.sort_values('count', ascending=False)
    
    print(f"Total unique participants: {len(klaatch_counts)}")
    print(f"Records per participant: min={klaatch_counts['count'].min()}, "
          f"max={klaatch_counts['count'].max()}, "
          f"mean={klaatch_counts['count'].mean():.1f}")
    
    # Initialize fold tracking
    fold_sizes = {i: 0 for i in range(n_splits)}
    fold_assignments = {}
    
    # Greedy assignment: assign to fold with least records
    for _, row in klaatch_counts.iterrows():
        klaatch_id = row['klaatch_id']
        count = row['count']
        
        # Find fold with minimum total records
        best_fold = min(fold_sizes, key=fold_sizes.get)
        
        # Assign participant to this fold
        fold_assignments[klaatch_id] = best_fold
        fold_sizes[best_fold] += count
    
    # Map assignments to DataFrame
    df['fold'] = df['klaatch_id'].map(fold_assignments)
    
    # Display fold distribution
    print("\nFold distribution:")
    for fold_id in sorted(fold_sizes.keys()):
        participants = sum(1 for k, v in fold_assignments.items() if v == fold_id)
        print(f"  Fold {fold_id}: {fold_sizes[fold_id]} records, {participants} participants")
    
    return df


# Assign folds with balanced distribution
df_with_folds = assign_folds_balanced(df, group_id_col='message_id', n_splits=N_SPLITS)

print(f"\n✓ Fold assignment complete")
print(f"Total records: {len(df_with_folds)}")

## 4. Validation: Verify No Data Leakage

Ensure that each participant (klaatch_id) appears in only one fold to prevent data leakage during cross-validation.

In [ ]:
def validate_fold_assignment(df, klaatch_id_col='klaatch_id', fold_col='fold'):
    """
    Validate that each klaatch_id is assigned to exactly one fold.
    
    This prevents data leakage during cross-validation by ensuring
    all records from the same participant stay together.
    
    Args:
        df (pd.DataFrame): DataFrame with klaatch_id and fold columns
        klaatch_id_col (str): Column name for participant ID
        fold_col (str): Column name for fold assignment
        
    Returns:
        dict: Dictionary of klaatch_ids appearing in multiple folds (empty if valid)
    """
    # Check unique folds per klaatch_id
    klaatch_fold_map = df.groupby(klaatch_id_col)[fold_col].nunique()
    
    # Identify problematic assignments
    invalid_assignments = klaatch_fold_map[klaatch_fold_map > 1].to_dict()
    
    if invalid_assignments:
        print("✗ VALIDATION FAILED!")
        print(f"Found {len(invalid_assignments)} participant(s) in multiple folds:")
        for klaatch_id, fold_count in invalid_assignments.items():
            folds = df[df[klaatch_id_col] == klaatch_id][fold_col].unique()
            print(f"  {klaatch_id}: appears in {fold_count} folds {list(folds)}")
        return invalid_assignments
    else:
        print("✓ VALIDATION PASSED: Each participant assigned to exactly one fold")
        
        # Additional statistics
        total_participants = df[klaatch_id_col].nunique()
        total_records = len(df)
        print(f"\nValidation statistics:")
        print(f"  Total participants: {total_participants}")
        print(f"  Total records: {total_records}")
        print(f"  Records per participant: {total_records / total_participants:.1f} avg")
        
        return {}


# Run validation
invalid_assignments = validate_fold_assignment(df_with_folds, KLAATCH_ID_COL, FOLD_COL)

## 5. Inspect Fold Assignments

Review the fold assignments to verify balance and correctness.

In [ ]:
# Display sample of fold assignments
print("Sample of fold assignments:")
print(df_with_folds[['fold', 'message_id', 'klaatch_id']].head(10))

print(f"\n{'-' * 60}")
print("Complete fold assignment data:")
df_with_folds[['fold', 'message_id', 'klaatch_id']]

## 6. Analyze Fold Distribution

Check the balance of records across folds and visualize the distribution.

In [ ]:
# Count records per fold
fold_counts = df_with_folds['fold'].value_counts().sort_index()

print("Records per fold:")
print(fold_counts)

# Calculate balance metrics
mean_count = fold_counts.mean()
std_count = fold_counts.std()
min_count = fold_counts.min()
max_count = fold_counts.max()

print(f"\nBalance metrics:")
print(f"  Mean: {mean_count:.1f} records/fold")
print(f"  Std Dev: {std_count:.2f}")
print(f"  Range: [{min_count}, {max_count}]")
print(f"  Difference: {max_count - min_count} records")
print(f"  Balance ratio: {min_count/max_count:.2%}")

# Visualize distribution
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
fold_counts.plot(kind='bar', color='steelblue', edgecolor='black')
plt.axhline(y=mean_count, color='red', linestyle='--', label=f'Mean: {mean_count:.1f}')
plt.xlabel('Fold')
plt.ylabel('Number of Records')
plt.title('Records Distribution Across Folds')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

fold_counts

## 7. Verify Participant Grouping

Check that all records from a specific participant are in the same fold.

In [ ]:
# Check a specific participant (example: 747)
example_klaatch_id = '747'

participant_data = df_with_folds[df_with_folds['klaatch_id'] == example_klaatch_id]

if len(participant_data) > 0:
    print(f"Participant {example_klaatch_id} details:")
    print(f"  Total records: {len(participant_data)}")
    print(f"  Assigned to fold: {participant_data['fold'].unique()[0]}")
    print(f"  All folds (should be 1 unique): {participant_data['fold'].nunique()}")
    
    print(f"\nSample records:")
    display_cols = ['KlaatchID', 'fold', 'message_id'] if 'KlaatchID' in participant_data.columns else ['klaatch_id', 'fold', 'message_id']
    print(participant_data[display_cols].head())
else:
    print(f"Participant {example_klaatch_id} not found in dataset")

# Show summary
participant_data[display_cols] if len(participant_data) > 0 else None

## 8. Update Database with Fold Assignments

Write the fold assignments back to the MySQL database.

In [ ]:
def update_database_with_folds(df, table_name, klaatch_id_col, fold_col, conn):
    """
    Update SQL table with klaatch_id and fold assignments.
    
    This function:
    1. Adds columns if they don't exist
    2. Updates each record with klaatch_id and fold values
    
    Args:
        df (pd.DataFrame): DataFrame with fold assignments
        table_name (str): Target table name
        klaatch_id_col (str): Column name for participant ID
        fold_col (str): Column name for fold assignment
        conn: MySQL connection object
    """
    cursor = conn.cursor()
    
    try:
        # Check and add klaatch_id column if needed
        cursor.execute(f"SHOW COLUMNS FROM {table_name} LIKE '{klaatch_id_col}'")
        if not cursor.fetchone():
            print(f"Adding column '{klaatch_id_col}' to table...")
            cursor.execute(f"ALTER TABLE {table_name} ADD COLUMN {klaatch_id_col} VARCHAR(255)")
        
        # Check and add fold column if needed
        cursor.execute(f"SHOW COLUMNS FROM {table_name} LIKE '{fold_col}'")
        if not cursor.fetchone():
            print(f"Adding column '{fold_col}' to table...")
            cursor.execute(f"ALTER TABLE {table_name} ADD COLUMN {fold_col} INT")
        
        # Update records with klaatch_id and fold values
        update_query = f"""
            UPDATE {table_name}
            SET {klaatch_id_col} = %s, {fold_col} = %s
            WHERE message_id = %s
        """
        
        print(f"\nUpdating {len(df)} records...")
        update_count = 0
        
        for _, row in df.iterrows():
            cursor.execute(update_query, (
                str(row['klaatch_id']),
                int(row['fold']),
                str(row['message_id'])
            ))
            update_count += 1
            
            # Progress indicator
            if update_count % 100 == 0:
                print(f"  Updated {update_count}/{len(df)} records...", end='\r')
        
        conn.commit()
        print(f"\n✓ Successfully updated {update_count} records in table '{table_name}'")
        
    except mysql.connector.Error as err:
        print(f"✗ Error updating database: {err}")
        conn.rollback()
        raise
    finally:
        cursor.close()


# Update the database
update_database_with_folds(
    df_with_folds,
    TABLE_NAME,
    KLAATCH_ID_COL,
    FOLD_COL,
    conn
)

# Close database connection
conn.close()
print("\n✓ Database connection closed")

## Summary

**Stratification Complete!**

The notebook has successfully:
1. ✓ Loaded data from MySQL database
2. ✓ Extracted participant IDs from message IDs
3. ✓ Assigned participants to balanced folds using greedy algorithm
4. ✓ Validated no data leakage (each participant in one fold only)
5. ✓ Updated database with fold assignments

**Key Features:**
- Balanced fold sizes (minimal variance)
- Participant-level grouping (prevents data leakage)
- Reproducible assignments
- Database persistence

**Usage for Cross-Validation:**
```python
# Example: Train/test split for fold 0
train_data = df[df['fold'] != 0]
test_data = df[df['fold'] == 0]
```

**Next Steps:**
- Use fold assignments for k-fold cross-validation
- Train models on k-1 folds, test on remaining fold
- Repeat for each fold to get robust performance estimates